In [1]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from IPython.display import display

# Önceki analizdeki proje konumunu koruyoruz.
PROJECT_ROOT = Path(
    "/Users/ilkinsoydas/Desktop/projeler/vscode/predictive-maintance"
)
HDF5_PATH = PROJECT_ROOT / "n_cmapss/data/N-CMAPSS_DS02-006.h5"

if not HDF5_PATH.is_file():
    raise FileNotFoundError(f"HDF5 dosyası bulunamadı: {HDF5_PATH}")


def decode_names(dataset):
    return [
        value.decode("utf-8") if isinstance(value, bytes) else str(value)
        for value in dataset[:]
    ]


with h5py.File(HDF5_PATH, mode="r") as h5:
    condition_names = decode_names(h5["W_var"])
    auxiliary_names = decode_names(h5["A_var"])

    assert "alt" in condition_names, "İrtifa değişkeni bulunamadı."
    assert {"unit", "cycle"}.issubset(auxiliary_names)
    assert h5["W_dev"].shape[0] == h5["A_dev"].shape[0]
    assert h5["W_dev"].shape[1] == len(condition_names)
    assert h5["A_dev"].shape[1] == len(auxiliary_names)

    print("Operasyonel koşullar:", condition_names)
    print("Yardımcı değişkenler:", auxiliary_names)
    print("W_dev boyutu:", h5["W_dev"].shape)
    print("A_dev boyutu:", h5["A_dev"].shape)
    print("İrtifa sütun indeksi:", condition_names.index("alt"))

    descriptions = {
        "alt": "İrtifa",
        "Mach": "Mach sayısı",
        "TRA": "Gaz kolu açısı",
        "T2": "Motor giriş toplam sıcaklığı",
    }

    display(pd.DataFrame({
        "Sütun indeksi": range(len(condition_names)),
        "Değişken": condition_names,
        "Açıklama": [
            descriptions.get(name, "Açıklama ayrıca incelenecek")
            for name in condition_names
        ],
    }))

    # Dosyada açık zaman bilgisi veya birim açıklaması var mı?
    print("\nDosya öznitelikleri:", dict(h5.attrs))
    print("W_dev öznitelikleri:", dict(h5["W_dev"].attrs))
    print("W_var öznitelikleri:", dict(h5["W_var"].attrs))
    print(
        "Adında time veya timestamp geçen kök nesneler:",
        [name for name in h5.keys()
         if "time" in name.lower() or "timestamp" in name.lower()],
    )

    # Yalnızca ilk 12 satırı okuyarak hizalamayı incele.
    sample_count = min(12, h5["W_dev"].shape[0])

    auxiliary_sample = pd.DataFrame(
        h5["A_dev"][:sample_count],
        columns=auxiliary_names,
    )
    conditions_sample = pd.DataFrame(
        h5["W_dev"][:sample_count],
        columns=condition_names,
    )

    sample = pd.concat(
        [auxiliary_sample, conditions_sample], axis=1
    )
    sample.insert(0, "hdf5_row", np.arange(sample_count))

print("\nİlk örnekler — hdf5_row zaman değil, dosyadaki satır indeksidir:")
display(sample)

Operasyonel koşullar: ['alt', 'Mach', 'TRA', 'T2']
Yardımcı değişkenler: ['unit', 'cycle', 'Fc', 'hs']
W_dev boyutu: (5263447, 4)
A_dev boyutu: (5263447, 4)
İrtifa sütun indeksi: 0


,Sütun indeksi,Değişken,Açıklama
0,0,alt,İrtifa
1,1,Mach,Mach sayısı
2,2,TRA,Gaz kolu açısı
3,3,T2,Motor giriş toplam sıcaklığı



Dosya öznitelikleri: {}
W_dev öznitelikleri: {}
W_var öznitelikleri: {}
Adında time veya timestamp geçen kök nesneler: []

İlk örnekler — hdf5_row zaman değil, dosyadaki satır indeksidir:


,hdf5_row,unit,cycle,Fc,hs,alt,Mach,TRA,T2
0,0,2.0,1.0,3.0,1.0,10005.0,0.448497,76.903748,502.420918
1,1,2.0,1.0,3.0,1.0,10013.0,0.447741,76.903748,502.326114
2,2,2.0,1.0,3.0,1.0,10017.0,0.448938,77.079529,502.416067
3,3,2.0,1.0,3.0,1.0,10024.0,0.449883,77.079529,502.469893
4,4,2.0,1.0,3.0,1.0,10031.0,0.449379,77.079529,502.401271
5,5,2.0,1.0,3.0,1.0,10040.0,0.447993,77.079529,502.248778
6,6,2.0,1.0,3.0,1.0,10044.0,0.448749,77.079529,502.297384
7,7,2.0,1.0,3.0,1.0,10048.0,0.449820,77.079529,502.376490
8,8,2.0,1.0,3.0,1.0,10056.0,0.448938,77.079529,502.269929
9,9,2.0,1.0,3.0,1.0,10059.0,0.448875,77.079529,502.254004
